# CV Final Project: Semi-Supervised Auto-Labeling and Edge-Deployable Model Training

**Students:**
- Omer Moshe Attia - 211398680
- Gavriel Levit - 207612417

---

## Project Overview

This project focuses on **semi-supervised dataset creation** and training a smaller model suitable for deployment on an **edge device**.

**Goals:**
- Create training data without human annotation (semi-supervised) using SAM3/Florence-2
- Select and train a model that can run on edge devices (Raspberry Pi, NVIDIA Jetson)
- Experience real-world challenges in object detection training

**Pipeline Summary:**
1. Filter Flickr30k by captions (31K → 10K images with persons/vehicles)
2. Auto-label with SAM3 and Florence-2
3. Validate labels via cross-model comparison, ground-truth evaluation, and TTA
4. Train edge-deployable detectors (YOLO26n, RetinaNet, MobileNetV4)

## Setup and Imports

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# Display settings
from IPython.display import Image, display, Markdown
import matplotlib.pyplot as plt
import pandas as pd

# Check CUDA availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---

# Part 1: Creating the Training Dataset

## 1.1 Dataset Filtering - Caption-Based Pre-Selection

Before running the auto-labelers, we filtered Flickr30k to select only images likely to contain **persons** or **vehicles**. Running SAM/Florence on all 31,000 images would waste compute on landscapes, food, and other irrelevant content.

### Method
Parse the Flickr30k captions CSV and search for keyword matches:

| Category | Keywords |
|----------|----------|
| **Person** | man, woman, person, child, elder, worker, workers, men, women, baby, infant, kid, kids, guy, girl, boy, female, male, people, pedestrian |
| **Vehicle** | car, truck, motorcycle, bike, scooter, van, jeep, moped, tractor, bus, semitrailer, vehicle, automobile, motorbike, bicycle, taxi, cab, suv, pickup |

### Selection Strategy: Balanced sampling to address vehicle scarcity
1. Take ALL images with vehicle keywords (vehicles are rare - only ~10% of matches)
2. Add person-only images at a 3:1 ratio to vehicles
3. Include 7% negative images (no keyword matches) to teach the model what's NOT a detection

In [ ]:
# Run caption-based filtering (skip if already done)
# !python scripts/filter_by_caption.py --strategy balanced --person-vehicle-ratio 3.0

# Display filtering results
filter_metadata = PROJECT_ROOT / "outputs" / "filtered_flickr" / "filter_metadata.txt"
if filter_metadata.exists():
    with open(filter_metadata, 'r') as f:
        header_lines = [next(f) for _ in range(12)]
    print("Filtering Results:")
    print("".join(header_lines))

### Filtering Results Summary

| Category | Count |
|----------|-------|
| Both (person + vehicle) | 2,869 |
| Person only | 6,212 |
| Vehicle only | 158 |
| Negative samples | 695 |
| **Total filtered** | **9,934** |

This reduced 31,000 images to ~10,000 relevant candidates before auto-labeling.

---

## 1.2 Auto-Labeling with SAM3 and Florence-2

We evaluated two state-of-the-art foundation models for auto-labeling:

| Model | Framework | Approach |
|-------|-----------|----------|
| **SAM 3** | Ultralytics | Text-prompted segmentation → bounding boxes |
| **Florence-2** | HuggingFace | Object detection + phrase grounding |

In [ ]:
# Run SAM3 labeling (skip if already done - takes several hours)
# !python scripts/run_sam_labeling.py --images-dir outputs/filtered_flickr/images

# Run Florence-2 labeling for comparison (skip if already done)
# !python scripts/run_florence_labeling.py --images-dir outputs/filtered_flickr/images

### Decisions and Rationale for Hyperparameters

| Parameter | Value | Rationale |
|-----------|-------|----------|
| **Confidence threshold** | 0.30 | Lower thresholds drown labels in noise; higher thresholds lose real detections in busy scenes. 0.30 is the precision/recall sweetspot. |
| **Min box size** | 1% of image area | Detections below 1% are probably noise or objects too small for the detector to learn from. |
| **Max box size** | 95% of image area | Boxes covering the entire image are false positives. 95% lets legitimate close-ups through. |
| **NMS IoU** | 0.50 | Industry-standard. Lower drops overlapping objects; higher leaves duplicates. |
| **Image size** | Original | Detection works better at native resolution. Small people get lost under aggressive downscaling. |

---

## 1.3 Evaluation Methodology

The auto-labeler was selected by running candidates through **five complementary evaluations**:

1. **Cross-model agreement & ensemble** - Compare SAM and Florence detections with IoU matching and Weighted Box Fusion (WBF)
2. **Ground-truth evaluation** - Run both models on Roboflow Persons & Cars dataset (2,057 human-labeled images)
3. **Sanity-check training** - Train YOLO26n for 50 epochs on a 50-image subset to verify labels are learnable
4. **Manual visual inspection** - Side-by-side visualization of disagreement images
5. **TTA validation** - Test-Time Augmentation consistency check

### 1.3.1 Cross-Model Agreement

We ran both SAM and Florence on the same 1,000 Flickr images and compared detections:

In [ ]:
# Cross-model comparison results
comparison_data = {
    'Metric': ['Matches at IoU > 0.5', 'Detections only in SAM', 'Detections only in Florence', 
               'Agreement rate', 'Average IoU on matched boxes'],
    'Value': ['2,796', '996', '705', '62.2%', '0.868']
}
df_comparison = pd.DataFrame(comparison_data)
display(Markdown("**Cross-Model Agreement Statistics:**"))
display(df_comparison.to_markdown(index=False))

In [ ]:
# Detection counts comparison
detection_data = {
    'Class': ['Person', 'Vehicle', 'Total'],
    'SAM 3': [3272, 520, 3792],
    'Florence-2': [2805, 696, 3501],
    'Δ': ['+467 (SAM)', '+176 (Florence)', '+291 (SAM)']
}
df_detections = pd.DataFrame(detection_data)
display(Markdown("**Detection Counts on Flickr30k (1,000 images):**"))
display(df_detections.to_markdown(index=False))

**Observation:** The models agree on 62.2% of detections. SAM is more conservative (fewer total detections) while Florence produces more vehicle detections but with higher false-positive rates.

### 1.3.2 Ground Truth Evaluation

Both auto-labelers were evaluated on the **Roboflow Persons & Cars dataset** (2,057 human-annotated images):

In [ ]:
# Ground truth evaluation results
gt_data = {
    'Metric': ['Precision', 'Recall', 'F1 score', 'mAP@0.5', 'mAP@0.5:0.95', 
               'True positives', 'False positives', 'False negatives'],
    'SAM 3': ['18.87%', '10.38%', '13.39%', '9.46%', '6.82%', '137', '589', '1,183'],
    'Florence-2': ['16.34%', '10.76%', '12.97%', '8.62%', '6.46%', '142', '727', '1,178'],
    'Winner': ['SAM (+15%)', 'Florence (+4%)', 'SAM (+3%)', 'SAM (+10%)', 'SAM (+6%)', 
               'Florence', 'SAM (-19%)', 'Florence']
}
df_gt = pd.DataFrame(gt_data)
display(Markdown("**Ground Truth Evaluation Results:**"))
display(df_gt.to_markdown(index=False))

**Key Insight:** The absolute numbers are low for both models, but **the relative comparison is the meaningful signal**: SAM wins on precision, F1, mAP, and false-positive rate; Florence edges SAM only on recall by a small margin.

### 1.3.3 Sanity-Check Training

A YOLO26n model was trained for 50 epochs on a 50-image subset from each label source. The goal was to test whether labels are **learnable**, not whether the model is good.

In [ ]:
# Sanity check results
sanity_data = {
    'Metric': ['Total train loss', 'Total val loss', 'Train/val gap (overfit ratio)', 
               'Precision', 'Recall', 'mAP@0.5'],
    'SAM': ['3.03', '5.67', '1.87×', '53.5%', '33.9%', '29.9%'],
    'Florence': ['2.74', '5.96', '2.17×', '48.0%', '40.5%', '38.0%'],
    'Ensemble (WBF)': ['2.84', '4.79', '1.68×', '81.1%', '32.9%', '42.6%']
}
df_sanity = pd.DataFrame(sanity_data)
display(Markdown("**Sanity-Check Training Results:**"))
display(df_sanity.to_markdown(index=False))

**Interpretation:** Naively, Florence scores better here. But we know from ground-truth tests that Florence has many false positives. The ensemble's higher precision (81.1%) and lower val loss shows that when we filter Florence's hallucinations via agreement, we get much better results. A dataset from Florence would be "learnable" - but would learn wrong things.

### 1.3.4 TTA (Test-Time Augmentation) Validation

**How TTA validates annotations:**

Test-Time Augmentation runs the same detection model multiple times on different transformed versions of the same image (horizontal flip, vertical flip). A **true object should be detected consistently** across augmentations.

**Implementation:**
1. Apply horizontal and vertical flips to each image
2. Run the labeler on each augmented version
3. Project detected boxes back to original coordinates
4. Compare with original labels using IoU matching
5. Classify boxes as "stable" (detected in multiple views) or "unstable" (inconsistent)

**Output metrics:**
- **Consistency score**: Average ratio of stable boxes per image
- **Stability rate**: Total stable boxes / total boxes
- **Low-consistency images**: Images where <50% of boxes are stable (flagged for review)

In [ ]:
# TTA Validation Results
tta_data = {
    'Metric': ['Images validated', 'Average consistency', 'Stability rate', 'Low-consistency images'],
    'SAM 3': ['200', '97.2%', '95.0%', '0'],
    'Florence-2': ['200', '94.8%', '91.2%', '3'],
    'Winner': ['-', 'SAM', 'SAM', 'SAM']
}
df_tta = pd.DataFrame(tta_data)
display(Markdown("**TTA Validation Results:**"))
display(df_tta.to_markdown(index=False))

**Result:** SAM's detections are highly stable across augmentations - 97.2% of boxes were detected consistently in multiple views, with **zero images flagged as low-consistency**. Florence showed slightly lower stability (94.8%) with 3 images where less than half of detections were reproducible.

### 1.3.5 Ensemble Validation

**How ensembles validate annotations:**

Ensemble validation runs multiple *different* detection models on the same image and treats **agreement between them as a proxy for correctness**. The motivation is that a single model's failure modes are systematic - Florence's tendency to invent vehicles, for example, will not be reproduced by SAM.

If two architecturally different models both place a box of the same class at the same location, the probability that both made the same wrong guess by coincidence is much lower than the probability that an object actually exists there.

We combined SAM 3 and Florence-2 outputs using **Weighted Box Fusion (WBF)**.

In [ ]:
# Ensemble agreement statistics
ensemble_data = {
    'Metric': ['Total images', 'Boxes both models agreed on', 'Boxes only SAM produced',
               'Boxes only Florence produced', 'Average per-image agreement score',
               'High-agreement images (≥80%)', 'Low-agreement images (<30%)'],
    'Value': ['1,000', '2,568', '871', '933', '61.0%', '309', '139']
}
df_ensemble = pd.DataFrame(ensemble_data)
display(Markdown("**Ensemble Agreement Statistics:**"))
display(df_ensemble.to_markdown(index=False))

---

## 1.4 Final Decision: SAM 3

In [ ]:
# Final decision matrix
decision_data = {
    'Criterion': ['Ground truth mAP', 'Precision', 'False positives', 'Manual inspection',
                  'TTA consistency', 'Recall', 'Vehicle detections', 'Sanity-check train loss',
                  'Sanity-check mAP', 'Final score'],
    'Weight': ['High', 'High', 'High', 'High', 'High', 'Medium', 'Medium', 'Low', 'Low', ''],
    'SAM 3': ['9.46%', '18.87%', '589', 'Reliable', '97.2%', '10.38%', '520', '3.03', '29.9%', '6'],
    'Florence-2': ['8.62%', '16.34%', '727', 'Many hallucinations', '94.8%', '10.76%', '696', '2.74', '38.0%', '3'],
    'Winner': ['SAM', 'SAM', 'SAM', 'SAM', 'SAM', 'Florence', 'Florence', 'Florence', 'Florence', 'SAM']
}
df_decision = pd.DataFrame(decision_data)
display(Markdown("**Final Decision Matrix:**"))
display(df_decision.to_markdown(index=False))

### Why SAM 3:
1. **Higher precision (18.9% vs 16.3%)**
2. **Fewer false positives (589 vs 727 - 23% lower)** - Training on SAM is safer
3. **Better mAP against ground truth (9.46% vs 8.62%)** - The most objective measure
4. **Manual inspection confirms quality** - SAM's failures are misses, Florence's are inventions

### Why NOT Florence-2:
1. **High false-positive rate** with consistent failure modes (animals → persons, scenes → vehicles)
2. **Sanity-check advantage is misleading** - Low training loss reproduces labels, doesn't validate them
3. **Quantity over quality is wrong** for semi-supervised pipelines

---

## 1.5 Prepare Final Dataset

In [ ]:
# Prepare train/val/test splits (skip if already done)
# !python scripts/prepare_dataset.py --labels-dir outputs/full_sam_labels/labels \
#     --images-dir outputs/filtered_flickr/images --output-dir outputs/datasets/sam_filtered

# Display dataset summary
dataset_yaml = PROJECT_ROOT / "outputs" / "datasets" / "sam_filtered" / "data.yaml"
if dataset_yaml.exists():
    import yaml
    with open(dataset_yaml, 'r') as f:
        data_config = yaml.safe_load(f)
    print("Dataset Configuration:")
    print(f"  Path: {data_config['path']}")
    print(f"  Classes: {data_config['names']}")
    print(f"  Number of classes: {data_config['nc']}")

### Final Dataset Summary

| Metric | Value |
|--------|-------|
| **Total images** | 9,380 |
| **Train / Val / Test** | 7,504 / 938 / 938 |
| **Total detections** | 44,274 |
| **Person detections** | 36,214 |
| **Vehicle detections** | 8,060 |
| **Class imbalance** | ~4.5:1 person-to-vehicle |

---

# Part 2: Training the Edge-Deployable Detector

## 2.1 Architecture Selection

We trained three architectures:

| Model | Framework | Rationale |
|-------|-----------|----------|
| **YOLO26n** | Ultralytics | Smallest YOLO variant, native edge export, primary deployment target |
| **RetinaNet** | Torchvision | Focal Loss addresses class imbalance; accuracy reference |
| **MobileNetV4** | Torchvision + timm | Advanced work - custom backbone integration |

### Why these three:
- **YOLO26n**: Deployment target (2.4M params, TFLite export)
- **RetinaNet**: Focal Loss designed for imbalanced datasets like ours (4.5:1)
- **MobileNetV4**: Fulfills assignment's advanced work requirement

### Why NOT others:
- **Faster R-CNN**: Two-stage, heavier, no accuracy advantage for two classes
- **EfficientDet**: Requires third framework for marginal gain

## 2.2 Training Configuration

| | YOLO26n | RetinaNet | MobileNetV4 |
|---|---------|-----------|-------------|
| Pretrained | COCO | COCO | ImageNet (backbone) |
| Optimizer | SGD | SGD | SGD |
| LR | 0.01 | 0.005 | 0.005 |
| Image size | 640 | 384 | 384 |
| Batch | 16 | 4 | 8 |
| Epochs | 50 | 50 | 50 |
| Hardware | RTX 4060 8GB | RTX 4060 8GB | RTX 4060 8GB |

In [ ]:
# Training commands (skip if already done - takes hours)

# YOLO26n baseline (no augmentation)
# !python scripts/train_yolo.py --data outputs/datasets/sam_filtered/data.yaml --model yolo26n.pt --phase baseline --epochs 50

# YOLO26n with augmentation
# !python scripts/train_yolo.py --data outputs/datasets/sam_filtered/data.yaml --model yolo26n.pt --phase augmented --epochs 50

# RetinaNet
# !python scripts/train_torchvision.py --data outputs/datasets/sam_filtered/data.yaml --model retinanet --epochs 50 --batch 4 --imgsz 384

# MobileNetV4 (advanced work)
# !python scripts/train_torchvision.py --data outputs/datasets/sam_filtered/data.yaml --model mobilenetv4 --epochs 50 --batch 8 --imgsz 384

## 2.3 Training Results

### YOLO26n Training Curves

In [ ]:
# Display YOLO26n augmented training curves
yolo_results_img = PROJECT_ROOT / "outputs" / "training" / "yolo26n_augmented" / "results.png"
if yolo_results_img.exists():
    display(Markdown("### YOLO26n Augmented - Training Curves"))
    display(Image(filename=str(yolo_results_img), width=1000))
else:
    print(f"Training results image not found at {yolo_results_img}")

**Analysis of Training Curves:**

1. **Train/Val Loss** - Both decreasing smoothly throughout training, tracking closely → **no overfitting**
2. **Classification Loss** - Drops sharply in early epochs, stabilizes → model learning class boundaries well
3. **Box Loss** - Steady decrease → bounding box regression improving
4. **mAP@50 and mAP@50-95** - Steadily climbing throughout → consistent learning
5. **Precision** - Some noise but trending upward → expected for detection tasks
6. **Recall** - Improving steadily → model finding more objects over time

In [ ]:
# Display confusion matrix
confusion_img = PROJECT_ROOT / "outputs" / "training" / "yolo26n_augmented" / "confusion_matrix.png"
if confusion_img.exists():
    display(Markdown("### YOLO26n Augmented - Confusion Matrix"))
    display(Image(filename=str(confusion_img), width=600))

In [ ]:
# Load and display YOLO training metrics
yolo_csv = PROJECT_ROOT / "outputs" / "training" / "yolo26n_augmented" / "results.csv"
if yolo_csv.exists():
    df_yolo = pd.read_csv(yolo_csv)
    df_yolo.columns = df_yolo.columns.str.strip()
    
    # Get final metrics
    final_row = df_yolo.iloc[-1]
    print("YOLO26n Augmented - Final Metrics (Epoch 50):")
    print(f"  Precision: {final_row['metrics/precision(B)']:.3f}")
    print(f"  Recall: {final_row['metrics/recall(B)']:.3f}")
    print(f"  mAP@0.5: {final_row['metrics/mAP50(B)']:.3f}")
    print(f"  mAP@0.5:0.95: {final_row['metrics/mAP50-95(B)']:.3f}")
    print(f"  Train Box Loss: {final_row['train/box_loss']:.3f}")
    print(f"  Val Box Loss: {final_row['val/box_loss']:.3f}")

### Baseline vs Augmented Comparison

In [ ]:
# Compare baseline vs augmented
baseline_csv = PROJECT_ROOT / "outputs" / "training" / "yolo26n_baseline" / "results.csv"
augmented_csv = PROJECT_ROOT / "outputs" / "training" / "yolo26n_augmented" / "results.csv"

if baseline_csv.exists() and augmented_csv.exists():
    df_baseline = pd.read_csv(baseline_csv)
    df_augmented = pd.read_csv(augmented_csv)
    df_baseline.columns = df_baseline.columns.str.strip()
    df_augmented.columns = df_augmented.columns.str.strip()
    
    baseline_final = df_baseline.iloc[-1]
    augmented_final = df_augmented.iloc[-1]
    
    comparison = {
        'Metric': ['Precision', 'Recall', 'mAP@0.5', 'mAP@0.5:0.95'],
        'Baseline': [f"{baseline_final['metrics/precision(B)']:.1%}",
                     f"{baseline_final['metrics/recall(B)']:.1%}",
                     f"{baseline_final['metrics/mAP50(B)']:.1%}",
                     f"{baseline_final['metrics/mAP50-95(B)']:.1%}"],
        'Augmented': [f"{augmented_final['metrics/precision(B)']:.1%}",
                      f"{augmented_final['metrics/recall(B)']:.1%}",
                      f"{augmented_final['metrics/mAP50(B)']:.1%}",
                      f"{augmented_final['metrics/mAP50-95(B)']:.1%}"],
        'Improvement': [
            f"+{(augmented_final['metrics/precision(B)'] - baseline_final['metrics/precision(B)'])*100:.1f}%",
            f"+{(augmented_final['metrics/recall(B)'] - baseline_final['metrics/recall(B)'])*100:.1f}%",
            f"+{(augmented_final['metrics/mAP50(B)'] - baseline_final['metrics/mAP50(B)'])*100:.1f}%",
            f"+{(augmented_final['metrics/mAP50-95(B)'] - baseline_final['metrics/mAP50-95(B)'])*100:.1f}%"
        ]
    }
    df_comp = pd.DataFrame(comparison)
    display(Markdown("**YOLO26n: Baseline vs Augmented:**"))
    display(df_comp.to_markdown(index=False))

### RetinaNet and MobileNetV4 Results

In [ ]:
import json

# Load RetinaNet results
retinanet_json = PROJECT_ROOT / "outputs" / "training" / "retinanet" / "retinanet_results.json"
if retinanet_json.exists():
    with open(retinanet_json, 'r') as f:
        retinanet_results = json.load(f)
    print("RetinaNet Results:")
    print(f"  Best F1: {retinanet_results['best_f1']:.3f} (epoch {retinanet_results['best_epoch']})")
    print(f"  Training time: {retinanet_results['total_time_minutes']:.1f} minutes")
    
    # Show epoch progression
    print("\n  Epoch progression:")
    for epoch_data in retinanet_results['epochs'][-3:]:
        print(f"    Epoch {epoch_data['epoch']}: F1={epoch_data['val_f1']:.3f}, "
              f"Precision={epoch_data['val_precision']:.3f}, Recall={epoch_data['val_recall']:.3f}")

In [ ]:
# Load MobileNetV4 results
mobilenet_json = PROJECT_ROOT / "outputs" / "training" / "mobilenetv4" / "mobilenetv4_results.json"
if mobilenet_json.exists():
    with open(mobilenet_json, 'r') as f:
        mobilenet_results = json.load(f)
    print("MobileNetV4 Results:")
    print(f"  Best F1: {mobilenet_results['best_f1']:.3f} (epoch {mobilenet_results['best_epoch']})")
    print(f"  Training time: {mobilenet_results['total_time_minutes']:.1f} minutes")
    
    # Show epoch progression
    print("\n  Epoch progression:")
    for epoch_data in mobilenet_results['epochs'][-3:]:
        print(f"    Epoch {epoch_data['epoch']}: F1={epoch_data['val_f1']:.3f}, "
              f"Precision={epoch_data['val_precision']:.3f}, Recall={epoch_data['val_recall']:.3f}")

### All Models Comparison

In [ ]:
# Summary comparison of all models
all_models = {
    'Model': ['YOLO26n (baseline)', 'YOLO26n (augmented)', 'RetinaNet', 'MobileNetV4'],
    'Precision': ['67.9%', '75.3%', '73.7%', '67.5%'],
    'Recall': ['48.2%', '50.9%', '62.2%', '44.8%'],
    'mAP@0.5': ['54.9%', '57.4%', '-', '-'],
    'F1': ['-', '-', '67.4%', '52.8%'],
    'Params': ['2.4M', '2.4M', '36.1M', '8.8M']
}
df_all = pd.DataFrame(all_models)
display(Markdown("**All Models Performance Comparison:**"))
display(df_all.to_markdown(index=False))

## 2.4 Training Insights

1. **Augmentation works.** YOLO26n gained ~+3-6% across metrics with standard augmentations (mosaic, horizontal flip, HSV jitter, scale jitter).

2. **Focal Loss handles class imbalance.** RetinaNet achieved the highest recall (62.2%) despite our 4.5:1 person-to-vehicle imbalance.

3. **MobileNetV4 underperformed.** The custom backbone integration worked, but needed more tuning. Lower metrics than both YOLO and RetinaNet.

4. **YOLO26n is the right choice for edge.** Best accuracy-to-size ratio (2.4M params), native TFLite export. RetinaNet is more accurate but 15× larger.

---

# Part 3: Inference

## 3.1 Single Image Inference

In [ ]:
# Run inference on a single image
from ultralytics import YOLO

# Load the best model
model_path = PROJECT_ROOT / "best_models" / "yolo26n_augmented.pt"
if model_path.exists():
    model = YOLO(str(model_path))
    print(f"Loaded model: {model_path}")
    print(f"Model classes: {model.names}")
else:
    print(f"Model not found at {model_path}")

In [ ]:
# Run inference on a sample image
sample_images = list((PROJECT_ROOT / "outputs" / "filtered_flickr" / "images").glob("*.jpg"))[:1]

if sample_images and model_path.exists():
    results = model.predict(str(sample_images[0]), conf=0.3, save=True, project=str(PROJECT_ROOT / "outputs"), name="notebook_inference")
    
    # Display results
    print(f"\nDetections on {sample_images[0].name}:")
    for r in results:
        for box in r.boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            cls_name = model.names[cls_id]
            print(f"  - {cls_name}: {conf:.2f}")

## 3.2 Batch Inference on Folder

In [ ]:
# Run inference on multiple images
# !python scripts/demo.py --image outputs/filtered_flickr/images --model yolo26n

# Or using the model directly:
if model_path.exists():
    sample_folder = PROJECT_ROOT / "outputs" / "filtered_flickr" / "images"
    sample_images = list(sample_folder.glob("*.jpg"))[:5]  # First 5 images
    
    if sample_images:
        print(f"Running inference on {len(sample_images)} images...")
        for img_path in sample_images:
            results = model.predict(str(img_path), conf=0.3, verbose=False)
            n_detections = sum(len(r.boxes) for r in results)
            print(f"  {img_path.name}: {n_detections} detections")

---

# Part 4: Model Export for Edge Deployment

In [ ]:
# Export to TFLite (for Raspberry Pi)
# !yolo export model=best_models/yolo26n_augmented.pt format=tflite imgsz=640

# Export to ONNX (for various runtimes)
# !yolo export model=best_models/yolo26n_augmented.pt format=onnx imgsz=640

# Check exported models
best_models_dir = PROJECT_ROOT / "best_models"
if best_models_dir.exists():
    print("Available models:")
    for model_file in sorted(best_models_dir.glob("*")):
        size_mb = model_file.stat().st_size / (1024 * 1024)
        print(f"  {model_file.name}: {size_mb:.1f} MB")

---

# Part 5: Conclusions and Reflections

## What We Learned

1. **Semi-supervised labeling is viable but requires validation** - SAM3 produced usable labels, but cross-validation with Florence-2 and ground-truth evaluation were essential to catch systematic errors.

2. **Class imbalance is a real challenge** - Our 4.5:1 person-to-vehicle ratio significantly affected vehicle detection. Focal Loss (RetinaNet) helped but didn't fully solve it.

3. **Augmentation provides consistent gains** - Simple augmentations (flip, scale, color jitter) improved YOLO26n by 3-6% across metrics.

4. **Edge deployment requires tradeoffs** - YOLO26n (2.4M params) vs RetinaNet (36M params) shows the accuracy-size tradeoff. For RPi deployment, YOLO26n is the right choice.

## Future Work

1. **Active learning** - Use model uncertainty to select which images to re-label with human review
2. **Class balancing** - Oversample vehicle images or use class-weighted loss
3. **Model distillation** - Distill RetinaNet knowledge into YOLO26n for better edge performance
4. **More augmentation** - MixUp, CutMix, and domain-specific augmentations

## What We Would Do Differently

1. Start with a smaller pilot (100 images) to validate the pipeline before scaling
2. Implement human-in-the-loop review for low-confidence detections
3. Train longer on RetinaNet to see if it continues improving

---

# Appendix: Running the Full Pipeline

```bash
# Step 1: Filter Flickr30k by captions
python scripts/filter_by_caption.py --strategy balanced --person-vehicle-ratio 3.0

# Step 2: Auto-labeling
python scripts/run_sam_labeling.py --images-dir outputs/filtered_flickr/images
python scripts/run_florence_labeling.py --images-dir outputs/filtered_flickr/images

# Step 3: Prepare dataset splits
python scripts/prepare_dataset.py --labels-dir outputs/full_sam_labels/labels \
    --images-dir outputs/filtered_flickr/images --output-dir outputs/datasets/sam_filtered

# Step 4: Train models
python scripts/train_yolo.py --data outputs/datasets/sam_filtered/data.yaml --model yolo26n.pt --phase baseline
python scripts/train_yolo.py --data outputs/datasets/sam_filtered/data.yaml --model yolo26n.pt --phase augmented
python scripts/train_torchvision.py --data outputs/datasets/sam_filtered/data.yaml --model retinanet --epochs 50
python scripts/train_torchvision.py --data outputs/datasets/sam_filtered/data.yaml --model mobilenetv4 --epochs 50

# Step 5: Export for edge
yolo export model=best_models/yolo26n_augmented.pt format=tflite imgsz=640

# Step 6: Inference
python scripts/demo.py --image path/to/image.jpg --model yolo26n
```